# 64 - Early Fusion on Primer conf60

**Arsitektur:** Early Fusion — landmark heatmap (Gaussian 224×224) di-stack sebagai channel ke-4 pada citra RGB → input 4-channel CNN.

**Referensi:** Wu et al. (MMM 2020) — *Emotion Recognition with Facial Landmark Heatmaps* (HAE-Net).

**Dataset:** Primer conf60 (6,795 samples, 37 users, confidence ≥ 60%).
**Backbone:** Scratch (`EmotionEarlyFusion`) + Transfer Learning (`EmotionEarlyFusionTransfer` dengan ResNet18).
**Skenario:** B1 (Baseline), B2 (Class Weights), B3 (Augmented — kalau dataset augmented tersedia).
**Kelas:** 7-class + 4-class remap.
**Metrik:** Macro / Micro / Weighted F1.

**Prerequisite:** heatmap file `X_{split}_heatmaps.npy` sudah digenerate via `scripts/generate_landmark_heatmaps.py`.

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionEarlyFusion, EmotionEarlyFusionTransfer
from training.utils import train_model, full_evaluation, get_class_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'frontonly_conf60' / 'early_fusion'
os.makedirs(OUTPUT_DIR, exist_ok=True)

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

In [ ]:
# ── Data loading: stack RGB image + landmark heatmap -> 4-channel ──

PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'


def load_split(split):
    """Load image + heatmap, return (N, 224, 224, 4) float32 concatenated tensor."""
    img = np.load(PRIMER_DIR / f'X_{split}_images.npy')       # (N, 224, 224, 3)
    heat = np.load(PRIMER_DIR / f'X_{split}_heatmaps.npy')    # (N, 224, 224)
    y = np.load(PRIMER_DIR / f'y_{split}.npy')
    if heat.ndim == 3:
        heat = heat[..., None]  # (N, 224, 224, 1)
    x4 = np.concatenate([img, heat], axis=-1)  # (N, 224, 224, 4)
    return x4.astype(np.float32, copy=False), y


X_tr, y_tr_7 = load_split('train')
X_v,  y_v_7  = load_split('val')
X_te, y_te_7 = load_split('test')

y_tr_4 = REMAP_4[y_tr_7]
y_v_4 = REMAP_4[y_v_7]
y_te_4 = REMAP_4[y_te_7]

print(f'Train 4-ch: {X_tr.shape}  Val: {X_v.shape}  Test: {X_te.shape}')
print(f'Train 7-class dist: {Counter(y_tr_7.tolist())}')
print(f'Train 4-class dist: {Counter(y_tr_4.tolist())}')

In [ ]:
# ── Helpers ──

def make_loader(x4, y, batch_size=32, shuffle=True):
    # (N, 224, 224, 4) -> (N, 4, 224, 224)
    t = torch.from_numpy(x4).permute(0, 3, 1, 2).contiguous()
    ys = torch.from_numpy(y).long()
    ds = TensorDataset(t, ys)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def train_and_eval(model_class, lr, tr_x, tr_y, v_x, v_y, te_x, te_y,
                   emotions, save_dir, class_weights=None):
    tr_loader = make_loader(tr_x, tr_y, BATCH_SIZE)
    val_loader = make_loader(v_x, v_y, BATCH_SIZE, False)
    test_loader = make_loader(te_x, te_y, BATCH_SIZE, False)

    model = model_class(num_classes=len(emotions)).to(device)
    save_path = str(save_dir / 'model.pth')

    criterion = nn.CrossEntropyLoss(weight=class_weights) if class_weights is not None \
                else nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)

    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, 'cnn', EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, 'cnn', emotions)
    return {
        'accuracy': float(r['test_accuracy']),
        'macro_f1': float(r['test_macro_f1']),
        'micro_f1': float(r['test_micro_f1']),
        'weighted_f1': float(r['test_weighted_f1']),
    }


def compute_class_weights(y, num_classes):
    """Inverse-frequency class weights."""
    counts = np.bincount(y, minlength=num_classes).astype(np.float32)
    counts = np.where(counts == 0, 1.0, counts)
    w = (1.0 / counts)
    w = w * num_classes / w.sum()
    return torch.from_numpy(w).to(device)


def run_early_fusion(num_classes, emotions,
                      tr_x, tr_y, v_x, v_y, te_x, te_y):
    print(f"\n{'='*70}")
    print(f'  EARLY FUSION — Primer conf60 ({num_classes}-class)')
    print(f"{'='*70}")
    results = {}
    cw = compute_class_weights(tr_y, num_classes)

    configs = [
        ('EarlyFusion_B1',   EmotionEarlyFusion,         0.0001, None),
        ('EarlyFusion_B2',   EmotionEarlyFusion,         0.0001, cw),
        ('EarlyFusion_TL_B1', EmotionEarlyFusionTransfer, 0.00005, None),
        ('EarlyFusion_TL_B2', EmotionEarlyFusionTransfer, 0.00005, cw),
    ]

    for key, model_cls, lr, class_weights in configs:
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)
        r = train_and_eval(model_cls, lr,
                           tr_x, tr_y, v_x, v_y, te_x, te_y,
                           emotions, save_dir,
                           class_weights=class_weights)
        results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")

    save_path = OUTPUT_DIR / f'early_fusion_{num_classes}c_results.json'
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return results

print('Helpers ready.')

## Run Early Fusion Experiments

In [ ]:
EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']

res_7c = run_early_fusion(7, EMOTIONS_7,
                            X_tr, y_tr_7, X_v, y_v_7, X_te, y_te_7)
res_4c = run_early_fusion(4, EMOTIONS_4,
                            X_tr, y_tr_4, X_v, y_v_4, X_te, y_te_4)

## Ringkasan Early Fusion (Semua Metrik)

In [ ]:
for nc_label, res in [('7-class', res_7c), ('4-class', res_4c)]:
    print(f"\n{'='*78}")
    print(f'  Early Fusion on Primer conf60 {nc_label} — sorted by Macro F1')
    print(f"{'='*78}")
    print(f"  {'Config':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for key in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[key]
        print(f"  {key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")